# ColBERT (2020)
[[paper]](https://arxiv.org/pdf/2004.04906)<br>
ColBERT = Contextualized Late Interaction over BERT

__ColBERT__ — реализация метода Dense Retrieval, предложенная исследователями из Стэнфордского университета в 2020 году, совмещающая точность cross-encoder модели и скорость two-tower моделей

__Постановка задачи__<br>
По заданному запросу $Q$ и корпусу документов $D = \{d_1, d_2, ..., d_N\}$, необходимо найти $K$ наиболее релевантных документов. Релевантность определяется таким образом, чтобы помочь в решении конечных задач, таких как Question Answering или обычный поиск информации.

__Мотивация__<br>
Арихтектура информационного поиска обычно балансирует между точностью и скоростью:
- ранняя интеракция (DPR, 2020) предполагает две отдельные нейронные сети, агрегирующие свои выходы скалярным произведением
    - выскокая скорость, так как векторы документов можно закэшировать в ANN-индексе (Approximate Nearest Neighbor)
    - низкая точность, так как есть "бутылочное горлышко"
- поздняя интеракция (BERT, 2018) предполагает конкатенацию запроса и документа в одном входе
    - высокая точность
    - низкая скорость, так как приходится выполнять полное вычисление

__Идея__<br>в быстрой двухбашенной архитектуре late interaction, вместо усреднения сигнала давайте использовать более выразительную MaxSim аггрегацию потокенных эмбедингов

<img src="img/colbert/colbert.png" width=500>

__Архитектура модели__<br>
ColBERT использует один <u>общий</u> BERT-энкодер для кодирования как запросов, так и документов. Ключевое отличие = вместо усреднения / использования `[CLS]`-токена, сохраняются все контекстуализированные векторные представления

1.  Токенизатор: стандартный WordPiece или BPE
2.  Encoder: BERT-подобная модель, возвращающая матрицу векторов $$E_Q = \{e_{q_1}, e_{q_2}, ..., e_{q_m}\}$$где $m$ — количество токенов в запросе ([CLS] в матрицуне добавляется)

__Алгоритм обучения__<br>
ColBERT обучается методом Contrastive Learning, как и DPR. Цель — научить модель давать высокие оценки релевантности для положительных пар (запрос, релевантный документ) и низкие для отрицательных пар (запрос, нерелевантный документ)

1.  Вход = тройка (запрос $Q$, положительный документ $D^+$, несколько отрицательных документов $D^-_i$)<br>при этом подчеркивается важность селекции хороших Hard Negatives - негативных примеров, близкзих к запросу, но тем не менее нерелевантных (с помощью BM25 / других Dense Retrieval моделей / путем майнинга из батча)<br><br>
2.  Кодируем BERT-энкодером ([CLS]-токены игнорируются. Для всех токен-векторов применяется L2-нормализация)$$E_Q = \text{BERT}(Q), \quad E_{D^+} = \text{BERT}(D^+), \quad E_{D^-_i} = \text{BERT}(D^-_i)$$
4.  Вычисление оценки релевантности __MaxSim__ для каждой пары (запрос, документ):
        $$Score(Q, D) = \sum_{e_q \in E_Q} \max_{e_d \in E_D} (e_q \cdot e_d)$$
    операция `max` ищет для каждого токена запроса наиболее похожий токен в документе (используя скалярное произведение, поскольку векторы L2-нормализованы, это эквивалентно косинусной близости). Затем эти максимальные схожести суммируются по всем токенам запроса. Это позволяет захватить "позднюю интеракцию", поскольку каждый токен запроса "взаимодействует" с каждым токеном документа<br><br>
5.  Cross-Entropy Loss:
        $$\mathcal{L} = -\log \frac{\exp(Score(Q, D^+))}{\sum_{D' \in \{D^+\} \cup \{D^-\}} \exp(Score(Q, D'))}$$максимизируем вероятность того, что положительный документ $D^+$ будет иметь наивысший балл релевантности среди всех рассматриваемых документов (положительного и отрицательных)<br><br>

__Построение индекса__
- для каждого документа $d_j$ пропускается через обученный ColBERT энкодер для получения его токен-векторов $E_{d_j}$
- эти наборы векторов $E_{d_j}$ (матрицы) сохраняются в специализированном индексе (например FAISS) или другие, адаптированные для эффективного поиска по MaxSim

__Инференс (поиск)__
1. запрос $Q$ пропускается через обученный ColBERT => токен-векторов $E_Q$
2. из быстрого ANN индекса достаем топ релевантных документов-кандидатов
3. для каждого $d_j$ из топа кандидатов вычисляем $Score(Q, d_j)$
4. возвращаем $K$ документов с наивысшими оценками релевантности

__Результаты__<br>
Тестировали на MS MARCO Passage Ranking и TREC Car. Основные выводы:
- прирост метрики MRR@10 на 10-15 процентных пунктов по сравнению с DPR (35% против 25%)
- в сотни раз быстрее, чем полноценные cross-encoders
- ColBERT должен хранить больше векторов на документ, чем Dense Retrieval, помогает квантизация векторов, используемая в ColBERTv2



## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример реализации основных концепций ColBERT на Python
# Используем библиотеки Hugging Face Transformers и FAISS для иллюстрации

from transformers import BertTokenizer, BertModel
import torch
import faiss
import numpy as np

# Инициализация токенизатора и модели BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Функция для кодирования текста в токен-векторы с использованием BERT
def encode_text(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    outputs = model(**inputs)
    # Отбрасываем [CLS] и [SEP] токены, берем только значимые токены
    token_embeddings = outputs.last_hidden_state[:, 1:-1, :]
    # L2-нормализация токен-векторов
    token_embeddings = torch.nn.functional.normalize(token_embeddings, p=2, dim=2)
    return token_embeddings.squeeze(0).detach().numpy()

# Пример документов и запросов
documents = [
    "The quick brown fox jumps over the lazy dog.",
    "A fast brown fox leaps over a sleepy dog.",
    "The quick red fox jumps over the lazy cat."
]

query = "A quick fox jumps over a lazy dog."

# Кодирование документов и запроса
document_embeddings = [encode_text(doc) for doc in documents]
query_embedding = encode_text(query)

# Индексация документов с использованием FAISS
dimension = document_embeddings[0].shape[1]
index = faiss.IndexFlatIP(dimension)  # Используем Inner Product (скалярное произведение)
for doc_embedding in document_embeddings:
    index.add(doc_embedding)

# Функция для вычисления MaxSim между запросом и документом
def maxsim(query_embedding, document_embedding):
    # Для каждого токена запроса находим максимальную схожесть с токенами документа
    max_similarities = np.max(np.dot(query_embedding, document_embedding.T), axis=1)
    # Суммируем максимальные схожести
    return np.sum(max_similarities)

# Поиск наиболее релевантного документа
scores = [maxsim(query_embedding, doc_embedding) for doc_embedding in document_embeddings]
best_doc_index = np.argmax(scores)

print(f"Most relevant document: {documents[best_doc_index]}")

# Вывод: "Most relevant document: The quick brown fox jumps over the lazy dog."
```

### Комментарии к коду:

1. **Токенизация и кодирование**: Используем `BertTokenizer` и `BertModel` из библиотеки Hugging Face для получения контекстуализированных токен-векторов. Мы отбрасываем `[CLS]` и `[SEP]` токены, как это делается в ColBERT, и нормализуем векторы.

2. **Индексация с FAISS**: FAISS используется для индексации токен-векторов документов. Это позволяет быстро находить документы с максимальной схожестью.

3. **MaxSim**: Реализуем операцию MaxSim, которая для каждого токена запроса находит максимальную схожесть с токенами документа и суммирует эти максимальные значения.

4. **Поиск**: Вычисляем MaxSim для каждого документа и выбираем документ с наивысшей оценкой как наиболее релевантный.

Этот пример иллюстрирует, как ColBERT использует многовекторное представление и позднюю интеракцию для достижения высокой точности в задачах информационного поиска.